# 02 — Registered baseline analysis

This notebook summarizes the immutable Week 2 CV baseline and the Week 4
comparison table. Historical holdout metrics are deliberately not loaded.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ANALYSIS_DIR = PROJECT_ROOT / "results/analysis/week4"
FIGURE_DIR = PROJECT_ROOT / "figures/week4"
OFFICIAL_DIR = PROJECT_ROOT / "results/hpo/ap_model_size_v1/full_run_01"


In [2]:
from src.week4_analysis import load_sources

sources = load_sources(PROJECT_ROOT)
baseline = pd.DataFrame([
    {
        "model": "Week 2 baseline",
        "cv_ap_mean": sources.baseline_cv["average_precision"]["mean"],
        "cv_ap_std": sources.baseline_cv["average_precision"]["std"],
        "full_train_serialized_mb": sources.baseline_model["file_size_mb"],
        "size_scope": "full-train pipeline",
    }
])
print(baseline.to_string(index=False))


          model  cv_ap_mean  cv_ap_std  full_train_serialized_mb          size_scope
Week 2 baseline    0.414861   0.007495                 14.008766 full-train pipeline


In [3]:
baseline_folds = pd.read_csv(PROJECT_ROOT / "results/baseline/fold_metrics.csv")
print(baseline_folds[["fold", "fit_samples", "validation_samples", "average_precision"]].to_string(index=False))


 fold  fit_samples  validation_samples  average_precision
    1        24112               12056           0.410533
    2        24112               12056           0.423515
    3        24112               12056           0.410535


## Baseline versus HPO reference points

![Baseline and key HPO models](../figures/week4/06_key_models_comparison.png)

The size panel states its measurement scope in every label. Baseline full-train
size must not be mislabeled as a maximum CV-fold size.


In [4]:
key_models = pd.read_csv(ANALYSIS_DIR / "key_models.csv")
print(key_models.to_string(index=False))


       model                    role  sampler  trial_number    config_hash  cv_ap_mean  cv_ap_std  cv_max_fold_size_mb  full_train_size_mb
    Baseline     registered_baseline baseline           NaN baseline_week2    0.414861   0.007495                  NaN           14.008766
 Best Random                 best_ap   random          31.0   ac0109f5352e    0.446163   0.004337             4.836080                 NaN
    Best Tpe                 best_ap      tpe          38.0   cd972e72ea0a    0.446186   0.004572             3.693059                 NaN
Selected TPE near_best_smallest_size      tpe          35.0   4cdd6dd35f1d    0.443417   0.005462             1.034871            1.289340
